## **<font color="blue">Define and Register Plugins</font>**
This section explains how to define Plugin classes and register them as part of your agent workflow.

### **Create Plugin Class**

In [1]:
# count_plugin.py

from google.adk.agents.base_agent import BaseAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.plugins.base_plugin import BasePlugin


class CountInvocationPlugin(BasePlugin):
  """A custom plugin that counts agent and tool invocations."""

  def __init__(self) -> None:
    """Initialize the plugin with counters."""
    super().__init__(name="count_invocation")
    self.agent_count: int = 0
    self.tool_count: int = 0
    self.llm_request_count: int = 0

  async def before_agent_callback(
      self, *, agent: BaseAgent, callback_context: CallbackContext
  ) -> None:
    """Count agent runs."""
    self.agent_count += 1
    print(f"[Plugin] Agent run count: {self.agent_count}")

  async def before_model_callback(
      self, *, callback_context: CallbackContext, llm_request: LlmRequest
  ) -> None:
    """Count LLM requests."""
    self.llm_request_count += 1
    print(f"[Plugin] LLM request count: {self.llm_request_count}")

### **Register Plugin Class**

In [2]:
# main.py

import asyncio

from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

# [Step 2] Import the plugin.
# from .count_plugin import CountInvocationPlugin


async def hello_world(tool_context: ToolContext, query: str):
  print(f'Hello world: query is [{query}]')


root_agent = Agent(
    model='gemini-2.0-flash',
    name='hello_world',
    description='Prints hello world with user query.',
    instruction="""Use hello_world tool to print hello world and user query.
    """,
    tools=[hello_world],
)


async def main():
  """Main entry point for the agent."""
  prompt = 'hello world'
  runner = InMemoryRunner(
      agent=root_agent,
      app_name='test_app_with_plugin',
      # [Step 2] Add your plugin here. You can add multiple plugins.
      plugins=[CountInvocationPlugin()],
  )
  session = await runner.session_service.create_session(
      user_id='user',
      app_name='test_app_with_plugin',
  )

  async for event in runner.run_async(
      user_id='user',
      session_id=session.id,
      new_message=types.Content(
          role='user', parts=[types.Part.from_text(text=prompt)]
      ),
  ):
    print(f'** Got event from {event.author}')


await main()
      
# if __name__ == '__main__':
#   asyncio.run(main())

[Plugin] Agent run count: 1
[Plugin] LLM request count: 1


D:\Agent-Development-Kit\venv\Lib\site-packages\google\adk\flows\llm_flows\base_llm_flow.py:449: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for event in agen:


** Got event from hello_world
Hello world: query is [hello world]
** Got event from hello_world
[Plugin] LLM request count: 2
** Got event from hello_world


## **<font color="blue">Build Workflows with Plugins</font>**
- A **hook** is just a specific point in a program where you can insert your own code. Think of it as a checkpoint or event in the workflow that “calls out” to let you run some code.
- Plugin callback hooks are a mechanism for implementing logic that intercepts, modifies, and even controls the agent's execution lifecycle. Each hook is a specific method in your Plugin class that you can implement to run code at a key moment. You have a choice between two modes of operation based on your hook's return value:
  - **To Observer: (No Return Value)**
    - Implement a hook with no return value (`None`). This approach is for tasks such as logging or collecting metrics, as it allows the agent's workflow to proceed to the next step without interruption.
    - For example, you could use `after_tool_callback` in a Plugin to log every tool's result for debugging.
    - Technical View:
      - Your hook doesn't return anythong (None).
      - The agent continues normally after your hook.
      - Use cases: Logging, monitoring, collecting metrics.
      - Exmple: `after_tool_callback` logs the output of every tool the agent uses.
        ```python
        def after_tool_callback(self, context, tool_result):
            print("Tool result:", tool_result)  # Just log it
            # No return -> workflow continues normally
        ```
  - **To Intervene: (Return a Value)**
    - Implement a hook and return a value. This approach short-circuits the workflow. The `Runner` halts processing, skip any subsequent plugins and the original intended action, like a Model call, and use a Plugin callback's return value as the result.
    - A common use case is implementing `before_model_callback` to return a cached `LlmResponse`, preventing a redundant and costly API call.
    - Technical View:
      - Your hook returns a value.
      - This stops the normal workflow immediately.
      - Skips any remaining plugins or the intended action (like calling the model).
      - Use case: Return a cached response instead of calling the model again.
        ```python
        def before_model_callback(self, context):
            if cached_result := get_cached_response(context.prompt):
                return cached_result  # Stops the model call, uses cached response
        ```
  - **To Amend: (Modify Context)**
    - Implement a hook and modify the Context object. This approach allows you to modify the context data for the module to be executed without otherwise interrupting the execution of that module.
    - For example, adding additional, standardized prompt text for Model object execution.
    - Technical View:
      - Your hook changes the context object.
      - Workflow continues, but the modules sees the modified context.
      - User case: Add extra instructions to a prompt before sending to a model.
        ```python
        def before_model_callback(self, context):
            context.prompt += "\nAlways answer in bullet points."  # Modify the prompt
            # No return -> workflow continues
        ```
- **Caution:**
  - Plugin callback funcitons have precedence over callbacks implemented at the object level. This behavior means that Any Plugin callbacks code is executed _before_ any Agent, Model, or Tool objects callbacks are executed. Futhermore, if a Plugin-level agent callback return any value, and not any empty(`None`) response, the Agent, Mode, or Tool-level callback is _not executed_ (skipped).
  - The Plugin desing establishes a hierarchy of code execution and separates global concerns from local agent logic. A Plugin is the stateful _module_ you build, such as `PerformanceMonitoringPlugin`, while the callback hooks are the specific _functions_ within that module that get executed. This architecture differs fundamentally from standard Agent Callbacks in these critical ways:
    - **Scope:** Plugin hooks are _global_. You register a Plugin once on the `Runner`, and its hooks apply universally to every Agent, Model, and Tool it manages. In contrast, Agent Callbacks are _local_, configured individually on a specific agent instance.
    - **Execution Order:** Plugins have _precedence_. For any given event, the Plugin hooks always run before any corresponding Agent Callback. This system behavior makes Plugins the correct architectural choice for implementing cross-cutting features like security policies, universal caching, and consistent logging across your entire application.


| Feature                  | Plugin Hooks (Global)                               | Agent Callbacks (Local)                     |
|---------------------------|---------------------------------------------------|--------------------------------------------|
| **Scope**                 | Global – applies to all Agents, Models, Tools   | Local – only applies to a specific Agent  |
| **Execution Order**       | Runs first, before any Agent/Model/Tool callbacks| Runs after Plugin hooks                     |
| **Return Value Behavior** | Returning a value **skips object-level callbacks**; returning `None` continues workflow | Always executed if workflow reaches it      |
| **Purpose**               | Cross-cutting concerns: logging, caching, security, monitoring | Custom agent-specific behavior             |
| **Stateful Module**       | Yes – can maintain metrics, cache, state        | Usually stateless                           |
| **Examples**              | `PerformanceMonitoringPlugin`, `SecurityPolicyPlugin` | Custom `before_call` or `after_call` on one agent |
| **Modes**                 | Observe (log), Intervene (short-circuit), Amend (modify context) | Mostly observe or amend agent behavior     |